# Hands-On: Text-to-Video Generation
**Prompt Engineering and Optimization for Gen AI (UE24AM343AB5)** · Unit 4

In this short notebook you will:
1. See that a video is just a **stack of images**
2. See why frames must be **consistent**
3. Write a good **video prompt**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

def play_video(frames, fps=4):
    """Play a stack of frames as a video inside the notebook."""
    fig, ax = plt.subplots(figsize=(3, 3))
    ax.axis("off")
    picture = ax.imshow(frames[0], cmap="gray", vmin=0, vmax=1)

    def show_frame(i):
        picture.set_data(frames[i])
        ax.set_title(f"Frame {i}")
        return [picture]

    anim = animation.FuncAnimation(fig, show_frame, frames=len(frames), interval=1000 / fps)
    plt.close(fig)                                       # stop a second, still copy appearing
    return HTML(anim.to_jshtml())

## 1. A video is a stack of images

An image is a grid of numbers. A video is many images in a row, one per **frame**.

Let's make a tiny video: a white square moving to the right, 6 frames long.

In [ ]:
frames = []

for step in range(6):
    image = np.zeros((20, 20))                          # a black 20x20 image
    image[8:12, step * 3 : step * 3 + 4] = 1            # draw a white square
    frames.append(image)

video = np.array(frames)
print("Video shape (frames, height, width):", video.shape)

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for i in range(6):
    axes[i].imshow(video[i], cmap="gray")
    axes[i].set_title(f"Frame {i}")
    axes[i].axis("off")
plt.show()

Now play the same 6 frames **one after another**. Press ▶ to watch the square move.

This is all a video is: frames shown quickly in order.

In [ ]:
play_video(video)

6 frames is very short. Let's make a **smoother** video with more, smaller steps.

In [ ]:
smooth_frames = []

for step in range(17):
    image = np.zeros((20, 20))
    image[8:12, step : step + 4] = 1                 # move 1 pixel each frame
    smooth_frames.append(image)

smooth_video = np.array(smooth_frames)
print("Video shape:", smooth_video.shape)

play_video(smooth_video, fps=8)

Real videos have **many** frames. From the slides: 5 seconds at 24 frames per second.

In [ ]:
seconds = 5
fps = 24

print("Frames to generate:", seconds * fps)

## 2. Frames must be consistent

If a model draws every frame **on its own**, the square jumps around randomly. This flicker is why video models make all frames **together**.

In [ ]:
random_frames = []

fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for i in range(6):
    image = np.zeros((20, 20))
    row, col = np.random.randint(0, 16, size=2)         # a random place each time
    image[row : row + 4, col : col + 4] = 1
    random_frames.append(image)
    axes[i].imshow(image, cmap="gray")
    axes[i].set_title(f"Frame {i}")
    axes[i].axis("off")
plt.show()

Now play these random frames as a video. Notice the **flicker**.

In [ ]:
play_video(np.array(random_frames))

Compare the two videos: the first moves **smoothly**, the second **jumps**. Keeping the same face, clothes and background in every frame is the hardest part of video generation.


### Save your video as a file

A video can be saved and shared like any file. Here we save the smooth video as a **GIF** (it opens in any browser). In Colab, find it in the **Files** panel on the left.

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
ax.axis("off")
picture = ax.imshow(smooth_video[0], cmap="gray", vmin=0, vmax=1)

def show_frame(i):
    picture.set_data(smooth_video[i])
    return [picture]

anim = animation.FuncAnimation(fig, show_frame, frames=len(smooth_video), interval=125)
anim.save("moving_square.gif", writer=animation.PillowWriter(fps=8))
plt.close(fig)

print("Saved: moving_square.gif")

## 3. Prompting for video

A video prompt needs what an image prompt needs, **plus** camera motion, sound and length.

In [ ]:
weak_prompt = "a dog on a beach"

shot     = "Wide shot at golden hour."
action   = "A golden retriever runs along the shoreline towards the camera, splashing water."
camera   = "Slow tracking shot."
style    = "Shallow depth of field, cinematic look."
sound    = "Sound: waves and distant gulls."
length   = "8 seconds, 16:9."

strong_prompt = " ".join([shot, action, camera, style, sound, length])

print("Weak  :", weak_prompt)
print("Strong:", strong_prompt)

**Your turn:** change the parts below to describe your own scene. Keep to **one action** and **one simple camera move**.

If you have a video tool (Veo in Gemini, Kling, Runway), try both the weak and your strong prompt and compare.

In [ ]:
shot   = "Close-up."
action = "A cup of coffee steams on a wooden table."
camera = "Slow dolly-in."
style  = "Warm morning light, cozy look."
sound  = "Sound: soft cafe chatter."
length = "5 seconds, 9:16."

my_prompt = " ".join([shot, action, camera, style, sound, length])
print(my_prompt)

## Key takeaways

- A video is a **stack of images** (frames) played in order. 5 s at 24 fps = **120 frames**.
- More frames with smaller changes = **smoother** motion.
- Frames made separately **flicker**. Video models generate frames **together** to keep them consistent.
- Good video prompts add **camera motion, sound and length** to subject, style and lighting.